# 01 - Bronze Ingestion

## Objetivo

Ler os arquivos CSV disponíveis na camada Landing e persistir os dados na camada Bronze em formato Delta Lake, mantendo os dados o mais próximo possível da origem.

## Entradas

- /Volumes/workspace/landing/raw_files/abi_bus_case1_beverage_sales_20210726.csv
- /Volumes/workspace/landing/raw_files/abi_bus_case1_beverage_channel_group_20210726.csv

## Saídas

- workspace.bronze.beverage_sales
- workspace.bronze.beverage_channel_group

In [0]:
# Importação das bibliotecas necessárias
from pyspark.sql.functions import *

In [0]:
# Definição dos caminhos dos arquivos na camada Landing
sales_file_path = "/Volumes/workspace/landing/raw_files/abi_bus_case1_beverage_sales_20210726.csv"
channel_file_path = "/Volumes/workspace/landing/raw_files/abi_bus_case1_beverage_channel_group_20210726.csv"

In [0]:
# Leitura do arquivo de vendas com separador TAB

sales_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", "\t")
    .option("encoding", "utf-8")
    .csv(sales_file_path)
)

display(sales_raw_df.limit(10))

��D A T E , C E _ B R A N D _ F L V R , B R A N D _ N M , B t l r _ O r g _ L V L _ C _ D e s c , C H N L _ G R O U P , T R A D E _ C H N L _ D E S C , P K G _ C A T , P k g _ C a t _ D e s c , T S R _ P C K G _ N M , $   V o l u m e , Y E A R , M O N T H , P E R I O D 
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , C A N A D A , L E I S U R E , S P O R T   V E N U E , N 2 0 O , 2 0 Z / 6 0 0 M L , . 5 9 1 L   N R P   2 4 L , 2 2 . 4 8 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , N O R T H E A S T , S U P E R S , S U P E R E T T E , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 1 0 0 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , S O U T H E A S T , W O R K P L A C E , P L A N T   /   O F F I C E , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 6 6 . 1 4 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 1 ,   R A S P B E R R Y , M I D W E S T , M A S S   M E R C H A N D I S E R , M A S S   M E R C H A N D I S E R , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 2 2 2 . 5 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , W E S T , M A S S   M E R C H A N D I S E R , M A S S   M E R C H A N D I S E R , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 3 0 2 . 5 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , M I D W E S T , O T H E R   S M A L L   S T O R E S , L I Q U O R / B E E R / W I N E / S O F T , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 1 0 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 1 ,   R A S P B E R R Y , S O U T H E A S T , C O N V E N I E N C E   R E T A I L , C O N V E N I E N C E   S T O R E , N 5 6 P , 5 0 0 M L   6 P , . 5 L   N R P   6 P , - 4 . 2 2 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , C A N A D A , F O O D   S E R V I C E , Q U I C K   S E R V I C E   R E S T A U R A N T , N 2 0 O , 2 0 Z / 6 0 0 M L , . 5 9 1 L   N R P   2 4 L , 5 9 . 9 5 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , S O U T H W E S T , S U P E R S , S U P E R M A R K E T , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 z   N R P   2 4 L   S , 3 0 0 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , S O U T H E A S T , F O O D   S E R V I C E , Q U I C K   S E R V I C E   R E S T A U R A N T , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 8 5 , 2 0 0 6 ,1.0,1.0


In [0]:
# Exibe o schema identificado pelo Spark após ajuste de separador

sales_raw_df.printSchema()

root
 |-- ��D A T E : string (nullable = true)
 |--  C E _ B R A N D _ F L V R : string (nullable = true)
 |--  B R A N D _ N M : string (nullable = true)
 |--  B t l r _ O r g _ L V L _ C _ D e s c : string (nullable = true)
 |--  C H N L _ G R O U P : string (nullable = true)
 |--  T R A D E _ C H N L _ D E S C : string (nullable = true)
 |--  P K G _ C A T : string (nullable = true)
 |--  P k g _ C a t _ D e s c : string (nullable = true)
 |--  T S R _ P C K G _ N M : string (nullable = true)
 |--  $   V o l u m e : string (nullable = true)
 |--  Y E A R : string (nullable = true)
 |--  M O N T H : double (nullable = true)
 |--  P E R I O D : double (nullable = true)



In [0]:
# Padronização dos nomes das colunas da base de vendas
# Remove espaços, caracteres especiais e corrige o nome da coluna de data

sales_raw_df = (
    sales_raw_df
    .withColumnRenamed(sales_raw_df.columns[0], "DATE")
    .withColumnRenamed("C E _ B R A N D _ F L V R", "CE_BRAND_FLVR")
    .withColumnRenamed("B R A N D _ N M", "BRAND_NM")
    .withColumnRenamed("B t l r _ O r g _ L V L _ C _ D e s c", "Btlr_Org_LVL_C_Desc")
    .withColumnRenamed("C H N L _ G R O U P", "CHNL_GROUP")
    .withColumnRenamed("T R A D E _ C H N L _ D E S C", "TRADE_CHNL_DESC")
    .withColumnRenamed("P K G _ C A T", "PKG_CAT")
    .withColumnRenamed("P k g _ C a t _ D e s c", "Pkg_Cat_Desc")
    .withColumnRenamed("T S R _ P C K G _ N M", "TSR_PCKG_NM")
    .withColumnRenamed("$   V o l u m e", "sales_volume")
    .withColumnRenamed("Y E A R", "YEAR")
    .withColumnRenamed("M O N T H", "MONTH")
    .withColumnRenamed("P E R I O D", "PERIOD")
)

sales_raw_df.printSchema()
display(sales_raw_df.limit(10))

root
 |-- DATE: string (nullable = true)
 |--  C E _ B R A N D _ F L V R : string (nullable = true)
 |--  B R A N D _ N M : string (nullable = true)
 |--  B t l r _ O r g _ L V L _ C _ D e s c : string (nullable = true)
 |--  C H N L _ G R O U P : string (nullable = true)
 |--  T R A D E _ C H N L _ D E S C : string (nullable = true)
 |--  P K G _ C A T : string (nullable = true)
 |--  P k g _ C a t _ D e s c : string (nullable = true)
 |--  T S R _ P C K G _ N M : string (nullable = true)
 |--  $   V o l u m e : string (nullable = true)
 |--  Y E A R : string (nullable = true)
 |--  M O N T H : double (nullable = true)
 |--  P E R I O D : double (nullable = true)



DATE, C E _ B R A N D _ F L V R , B R A N D _ N M , B t l r _ O r g _ L V L _ C _ D e s c , C H N L _ G R O U P , T R A D E _ C H N L _ D E S C , P K G _ C A T , P k g _ C a t _ D e s c , T S R _ P C K G _ N M , $   V o l u m e , Y E A R , M O N T H , P E R I O D 
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , C A N A D A , L E I S U R E , S P O R T   V E N U E , N 2 0 O , 2 0 Z / 6 0 0 M L , . 5 9 1 L   N R P   2 4 L , 2 2 . 4 8 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , N O R T H E A S T , S U P E R S , S U P E R E T T E , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 1 0 0 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , S O U T H E A S T , W O R K P L A C E , P L A N T   /   O F F I C E , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 6 6 . 1 4 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 1 ,   R A S P B E R R Y , M I D W E S T , M A S S   M E R C H A N D I S E R , M A S S   M E R C H A N D I S E R , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 2 2 2 . 5 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , W E S T , M A S S   M E R C H A N D I S E R , M A S S   M E R C H A N D I S E R , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 3 0 2 . 5 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , M I D W E S T , O T H E R   S M A L L   S T O R E S , L I Q U O R / B E E R / W I N E / S O F T , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 1 0 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 1 ,   R A S P B E R R Y , S O U T H E A S T , C O N V E N I E N C E   R E T A I L , C O N V E N I E N C E   S T O R E , N 5 6 P , 5 0 0 M L   6 P , . 5 L   N R P   6 P , - 4 . 2 2 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , C A N A D A , F O O D   S E R V I C E , Q U I C K   S E R V I C E   R E S T A U R A N T , N 2 0 O , 2 0 Z / 6 0 0 M L , . 5 9 1 L   N R P   2 4 L , 5 9 . 9 5 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , S O U T H W E S T , S U P E R S , S U P E R M A R K E T , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 z   N R P   2 4 L   S , 3 0 0 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , S O U T H E A S T , F O O D   S E R V I C E , Q U I C K   S E R V I C E   R E S T A U R A N T , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 8 5 , 2 0 0 6 ,1.0,1.0


In [0]:
# Leitura do arquivo de agrupamento de canais
# Esta base utiliza vírgula como separador

channel_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ",")
    .option("encoding", "utf-8")
    .csv(channel_file_path)
)

display(channel_raw_df.limit(10))
channel_raw_df.printSchema()

TRADE_CHNL_DESC,TRADE_GROUP_DESC,TRADE_TYPE_DESC
SPORT VENUE,ENTERTAINMENT,ALCOHOLIC
SUPERETTE,SERVICES,MIX
PLANT / OFFICE,SERVICES,MIX
MASS MERCHANDISER,GROCERY,MIX
LIQUOR/BEER/WINE/SOFT,GROCERY,ALCOHOLIC
CONVENIENCE STORE,SERVICES,MIX
QUICK SERVICE RESTAURANT,ENTERTAINMENT,ALCOHOLIC
SUPERMARKET,GROCERY,MIX
ALL OTHER,OTHER,MIX
OTHER EATING + DRINKING,SERVICES,MIX


root
 |-- TRADE_CHNL_DESC: string (nullable = true)
 |-- TRADE_GROUP_DESC: string (nullable = true)
 |-- TRADE_TYPE_DESC: string (nullable = true)



In [0]:
# Padronização dos nomes das colunas da base de vendas
# Remove espaços, caracteres especiais e corrige o nome da coluna de data

sales_raw_df = (
    sales_raw_df
    .withColumnRenamed(sales_raw_df.columns[0], "DATE")
    .withColumnRenamed("C E _ B R A N D _ F L V R", "CE_BRAND_FLVR")
    .withColumnRenamed("B R A N D _ N M", "BRAND_NM")
    .withColumnRenamed("B t l r _ O r g _ L V L _ C _ D e s c", "Btlr_Org_LVL_C_Desc")
    .withColumnRenamed("C H N L _ G R O U P", "CHNL_GROUP")
    .withColumnRenamed("T R A D E _ C H N L _ D E S C", "TRADE_CHNL_DESC")
    .withColumnRenamed("P K G _ C A T", "PKG_CAT")
    .withColumnRenamed("P k g _ C a t _ D e s c", "Pkg_Cat_Desc")
    .withColumnRenamed("T S R _ P C K G _ N M", "TSR_PCKG_NM")
    .withColumnRenamed("$   V o l u m e", "sales_volume")
    .withColumnRenamed("Y E A R", "YEAR")
    .withColumnRenamed("M O N T H", "MONTH")
    .withColumnRenamed("P E R I O D", "PERIOD")
)

sales_raw_df.printSchema()
display(sales_raw_df.limit(10))

root
 |-- DATE: string (nullable = true)
 |-- CE_BRAND_FLVR: string (nullable = true)
 |-- BRAND_NM: string (nullable = true)
 |-- Btlr_Org_LVL_C_Desc: string (nullable = true)
 |-- CHNL_GROUP: string (nullable = true)
 |-- TRADE_CHNL_DESC: string (nullable = true)
 |-- PKG_CAT: string (nullable = true)
 |-- Pkg_Cat_Desc: string (nullable = true)
 |-- TSR_PCKG_NM: string (nullable = true)
 |-- sales_volume: string (nullable = true)
 |-- YEAR: string (nullable = true)
 |-- MONTH: double (nullable = true)
 |-- PERIOD: double (nullable = true)



DATE,CE_BRAND_FLVR,BRAND_NM,Btlr_Org_LVL_C_Desc,CHNL_GROUP,TRADE_CHNL_DESC,PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,sales_volume,YEAR,MONTH,PERIOD
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , C A N A D A , L E I S U R E , S P O R T   V E N U E , N 2 0 O , 2 0 Z / 6 0 0 M L , . 5 9 1 L   N R P   2 4 L , 2 2 . 4 8 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , N O R T H E A S T , S U P E R S , S U P E R E T T E , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 1 0 0 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , S O U T H E A S T , W O R K P L A C E , P L A N T   /   O F F I C E , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 6 6 . 1 4 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 1 ,   R A S P B E R R Y , M I D W E S T , M A S S   M E R C H A N D I S E R , M A S S   M E R C H A N D I S E R , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 2 2 2 . 5 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , W E S T , M A S S   M E R C H A N D I S E R , M A S S   M E R C H A N D I S E R , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 3 0 2 . 5 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , M I D W E S T , O T H E R   S M A L L   S T O R E S , L I Q U O R / B E E R / W I N E / S O F T , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 1 0 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 4 4 1 ,   R A S P B E R R Y , S O U T H E A S T , C O N V E N I E N C E   R E T A I L , C O N V E N I E N C E   S T O R E , N 5 6 P , 5 0 0 M L   6 P , . 5 L   N R P   6 P , - 4 . 2 2 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , C A N A D A , F O O D   S E R V I C E , Q U I C K   S E R V I C E   R E S T A U R A N T , N 2 0 O , 2 0 Z / 6 0 0 M L , . 5 9 1 L   N R P   2 4 L , 5 9 . 9 5 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , S O U T H W E S T , S U P E R S , S U P E R M A R K E T , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 z   N R P   2 4 L   S , 3 0 0 , 2 0 0 6 ,1.0,1.0
 1 / 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , S O U T H E A S T , F O O D   S E R V I C E , Q U I C K   S E R V I C E   R E S T A U R A N T , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 8 5 , 2 0 0 6 ,1.0,1.0


In [0]:
# Correção definitiva dos nomes das colunas para compatibilidade com Delta Lake

sales_raw_df = sales_raw_df.toDF(
    "DATE",
    "CE_BRAND_FLVR",
    "BRAND_NM",
    "Btlr_Org_LVL_C_Desc",
    "CHNL_GROUP",
    "TRADE_CHNL_DESC",
    "PKG_CAT",
    "Pkg_Cat_Desc",
    "TSR_PCKG_NM",
    "sales_volume",
    "YEAR",
    "MONTH",
    "PERIOD"
)

print("Colunas após padronização:")
print(sales_raw_df.columns)

Colunas após padronização:
['DATE', 'CE_BRAND_FLVR', 'BRAND_NM', 'Btlr_Org_LVL_C_Desc', 'CHNL_GROUP', 'TRADE_CHNL_DESC', 'PKG_CAT', 'Pkg_Cat_Desc', 'TSR_PCKG_NM', 'sales_volume', 'YEAR', 'MONTH', 'PERIOD']


In [0]:
# Releitura correta do arquivo de vendas
# O arquivo de vendas está em UTF-16 e separado por TAB

sales_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", "\t")
    .option("encoding", "utf-16")
    .csv(sales_file_path)
)

sales_raw_df = sales_raw_df.withColumnRenamed("$ Volume", "sales_volume")

display(sales_raw_df.limit(10))
sales_raw_df.printSchema()

DATE,CE_BRAND_FLVR,BRAND_NM,Btlr_Org_LVL_C_Desc,CHNL_GROUP,TRADE_CHNL_DESC,PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,sales_volume,YEAR,MONTH,PERIOD
1/1/2006,3440,LEMON,CANADA,LEISURE,SPORT VENUE,N20O,20Z/600ML,.591L NRP 24L,22.48,2006,1,1
1/1/2006,3440,LEMON,NORTHEAST,SUPERS,SUPERETTE,N20O,20Z/600ML,20Z NRP 24L,100.0,2006,1,1
1/1/2006,3554,STRAWBERRY,SOUTHEAST,WORKPLACE,PLANT / OFFICE,N20O,20Z/600ML,20Z NRP 24L,66.14,2006,1,1
1/1/2006,3441,RASPBERRY,MIDWEST,MASS MERCHANDISER,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,222.5,2006,1,1
1/1/2006,3440,LEMON,WEST,MASS MERCHANDISER,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,302.5,2006,1,1
1/1/2006,3440,LEMON,MIDWEST,OTHER SMALL STORES,LIQUOR/BEER/WINE/SOFT,N20O,20Z/600ML,20Z NRP 24L,10.0,2006,1,1
1/1/2006,3441,RASPBERRY,SOUTHEAST,CONVENIENCE RETAIL,CONVENIENCE STORE,N56P,500ML 6P,.5L NRP 6P,-4.22,2006,1,1
1/1/2006,3554,STRAWBERRY,CANADA,FOOD SERVICE,QUICK SERVICE RESTAURANT,N20O,20Z/600ML,.591L NRP 24L,59.95,2006,1,1
1/1/2006,3554,STRAWBERRY,SOUTHWEST,SUPERS,SUPERMARKET,N20O,20Z/600ML,20z NRP 24L S,300.0,2006,1,1
1/1/2006,3554,STRAWBERRY,SOUTHEAST,FOOD SERVICE,QUICK SERVICE RESTAURANT,N20O,20Z/600ML,20Z NRP 24L,85.0,2006,1,1


root
 |-- DATE: string (nullable = true)
 |-- CE_BRAND_FLVR: integer (nullable = true)
 |-- BRAND_NM: string (nullable = true)
 |-- Btlr_Org_LVL_C_Desc: string (nullable = true)
 |-- CHNL_GROUP: string (nullable = true)
 |-- TRADE_CHNL_DESC: string (nullable = true)
 |-- PKG_CAT: string (nullable = true)
 |-- Pkg_Cat_Desc: string (nullable = true)
 |-- TSR_PCKG_NM: string (nullable = true)
 |-- sales_volume: double (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- PERIOD: string (nullable = true)



In [0]:
# Persistência das tabelas na camada Bronze

sales_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.beverage_sales")

channel_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.beverage_channel_group")

print("Tabelas Bronze criadas com sucesso.")

Tabelas Bronze criadas com sucesso.


In [0]:
# Validação da camada Bronze

print("Quantidade de registros - Beverage Sales:")
print(spark.table("workspace.bronze.beverage_sales").count())

print("Quantidade de registros - Beverage Channel Group:")
print(spark.table("workspace.bronze.beverage_channel_group").count())

Quantidade de registros - Beverage Sales:
16151
Quantidade de registros - Beverage Channel Group:
31
